# Full-cohort held-out risk scores across all schemes/events

Calls `run_full_cohort_risk_scores.py` for every `(scheme, event)` pair whose full-cohort CV training has completed (i.e. `text_val.csv.gz` exists). Generates per-patient held-out risk scores for the text and base models so they can be compared on the same cohort (e.g. stratified KM curves).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from tqdm.auto import tqdm

from slurm_array_utils import SCHEME_CONFIG, get_output_dir

In [ ]:
SCRIPT = Path("run_full_cohort_risk_scores.py").resolve()
PYTHON = sys.executable

# Subset of schemes to run; default = all four
SCHEMES = sorted(SCHEME_CONFIG.keys())

# Pass-through flags to the worker script
OVERWRITE = False
N_JOBS = None       # None => SLURM_CPUS_PER_TASK env var, falls back to 1
MAX_ITER = 1000
BACKEND = "threading"

print(f"Script: {SCRIPT}")
print(f"Schemes: {SCHEMES}")

## Discover completed (scheme, event) pairs

Only events whose full-cohort training has produced `text_val.csv.gz` are eligible — those CV results are what the script reads to pick the best hyperparameters.

In [ ]:
def _trained_events(scheme: str) -> list[str]:
    train_root = get_output_dir(scheme, "full_cohort")
    if not os.path.isdir(train_root):
        return []
    return sorted(
        ev for ev in os.listdir(train_root)
        if os.path.exists(os.path.join(train_root, ev, "text_val.csv.gz"))
    )

def _risk_done(scheme: str, event: str) -> bool:
    risk_dir = os.path.join(get_output_dir(scheme, "full_cohort_risk_scores"), event)
    return (
        os.path.exists(os.path.join(risk_dir, "text_risk_scores.csv.gz"))
        and os.path.exists(os.path.join(risk_dir, "base_risk_scores.csv.gz"))
    )

tasks: list[tuple[str, str]] = []
for scheme in SCHEMES:
    events = _trained_events(scheme)
    pending = [ev for ev in events if OVERWRITE or not _risk_done(scheme, ev)]
    skipped = len(events) - len(pending)
    print(f"{scheme}: {len(events)} trained, {len(pending)} pending ({skipped} already have risk scores)")
    tasks.extend((scheme, ev) for ev in pending)

print(f"\nTotal tasks queued: {len(tasks)}")

## Run

Each task is invoked as a subprocess so a failure on one event does not kill the loop. Per-task stdout is captured and only printed for failures.

In [ ]:
def _build_cmd(scheme: str, event: str) -> list[str]:
    cmd = [PYTHON, str(SCRIPT), "--scheme", scheme, "--event", event,
           "--max-iter", str(MAX_ITER), "--backend", BACKEND]
    if OVERWRITE:
        cmd.append("--overwrite")
    if N_JOBS is not None:
        cmd += ["--n-jobs", str(N_JOBS)]
    return cmd

results = {"ok": [], "failed": []}
for scheme, event in tqdm(tasks, desc="events"):
    proc = subprocess.run(_build_cmd(scheme, event), capture_output=True, text=True)
    if proc.returncode == 0:
        results["ok"].append((scheme, event))
    else:
        results["failed"].append((scheme, event, proc.returncode))
        print(f"\n[FAIL] {scheme}:{event} (exit {proc.returncode})")
        if proc.stdout:
            print("--- stdout ---")
            print(proc.stdout)
        if proc.stderr:
            print("--- stderr ---")
            print(proc.stderr)

print(f"\nDone. {len(results['ok'])} succeeded, {len(results['failed'])} failed.")

## Summary

In [ ]:
for scheme in SCHEMES:
    n_done = sum(1 for ev in _trained_events(scheme) if _risk_done(scheme, ev))
    n_train = len(_trained_events(scheme))
    print(f"{scheme}: {n_done}/{n_train} events have risk scores")

if results["failed"]:
    print("\nFailures:")
    for scheme, event, rc in results["failed"]:
        print(f"  {scheme}:{event} (exit {rc})")